In [ ]:
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input

# a. Loading and preprocessing the image data
print("--- Stage A: Loading and preprocessing ---")
# Load the dataset from CSV files
train_data = pd.read_csv('sign_mnist_train.csv')
test_data = pd.read_csv('sign_mnist_test.csv')

# Separate labels (answers) and features (pixels)
y_train = train_data['label'].values
x_train = train_data.drop('label', axis=1).values

y_test = test_data['label'].values
x_test = test_data.drop('label', axis=1).values

# Normalize pixel values to 0-1 scale
x_train = x_train / 255.0
x_test = x_test / 255.0

# CRITICAL FOR CNN: Reshape from a flat 784 array to a 28x28 grid with 1 grayscale channel
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# b. Defining the model's architecture
print("\n--- Stage B: Defining the architecture ---")
model = Sequential([
    Input(shape=(28, 28, 1)),
    
    # Convolutional blocks to extract features (like finger positions and hand shapes)
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Flatten and Classify
    Flatten(),
    Dense(64, activation='relu'),
    # 25 output neurons because labels go from 0 to 24 (A-Z, skipping J and Z)
    Dense(25, activation='softmax') 
])

# c. Training the model
print("\n--- Stage C: Training the model ---")
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 5 epochs is generally enough for a CNN to learn this dataset well
history = model.fit(
    x_train, y_train, 
    epochs=5, 
    batch_size=64, 
    validation_data=(x_test, y_test)
)

# d. Estimating the model's performance
print("\n--- Stage D: Estimating performance ---")
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Final Test Loss: {test_loss:.4f}")
print(f"Final Test Accuracy: {test_accuracy * 100:.2f}%")

# Plotting the performance metrics
plt.figure(figsize=(12, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss (Sign Language)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy (Sign Language)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()